# Definition of Ready (DoR) Agent

**The problem:** a story that reaches "To Do" without enough to build from - maybe no
acceptance criteria or problem statement - stalls out.
That's costly for offshore devs and other agents picking up work
asynchronously: there's no synchronous handoff to just ask someone in the same
timezone, so a gap that would cost minutes in person risks a full day of waiting.

**Goal:** catch that gap before a human (or agent) loses a cycle to it. Every card in
"To Do" should already say whether it's ready to build, and why - so anyone can pick it
up, in any timezone, without waiting on a synchronous handoff.

**How:** a run-to-completion, episodic agent - one bounded pass per story: fetch,
reason, escalate through tools only as far as actually needed, reach a verdict, take a
terminal action, stop. Nothing carries over between stories or between runs.

**Built on:** the [Shortcut](https://shortcut.com) API for stories, epics, labels and
comments - the workflow this agent operates on. Where a story's detailed spec or reference implementation lives in a
linked Google Doc rather than the card description or discussion, the agent accesses it directly via
the Google Drive API (read-only, service account).

A few key design aspects (more detail down in "the agent" section):

- **Explainability** - every verdict carries a stated reason, posted where the team already looks.
- **Portability, for cost control** - swapping models is a one-line change; per-model quirks are isolated so a swap never breaks the agent logic.
- **Token-frugal by design** - tools are weighted by cost and impact, so the agent only reaches for an expensive one once a cheaper one has failed to answer the question.


In [ ]:
import os
import requests
import arrow
import re
import io
import base64
import mimetypes
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError
from pypdf import PdfReader
from docx import Document
import anthropic
from anthropic import beta_tool

# works over the workspace graph to validate that a story has enough information to be worked on 
# "is it a viable story" is an example - the agent can be prepped with any business logic

# costs ~2c for a complex story

# a read-and-comment token assigned to the agent's account - set SHORTCUT_TOKEN before starting the kernel
token = os.environ['SHORTCUT_TOKEN']

headers = {
    'Content-Type': 'application/json',
    'Shortcut-Token': token
}



api_base = "https://api.app.shortcut.com/api"
    

# get the agent's user_id 
who_am_i = requests.get("https://api.app.shortcut.com/api/v3/member", headers=headers).json()
authenticated_user_id = who_am_i['id']


In [ ]:
"""
configure this against your target state - example is a default state named "To Do"
"""

# a quick search to get the "TO DO" (i.e. prioritised but pre-picked up)

def search_stories(target_state_name='To Do', days_old=10):
    
    # get todo
    response = requests.get(f'{api_base}/v3/workflows', headers=headers)


    target_col = list(i for i in response.json()[0]['states'] if i['name'] == target_state_name)[0]

    target_state_id = target_col['id']

    print(f"💬 retrieving stories with state {target_state_id}")
    
    """
    this gets the stories within the workflow state created or updated within the last 10 days
    """

    json_data = {
        'archived': False,
        # 'updated_at_start': '2016-12-31T12:30:00Z', # one week ago
        'updated_at_start': str(arrow.now().shift(days=-days_old)),
        'workflow_state_id': target_state_id,
        'workflow_state_types': [
            'backlog',
        ],
    }

    response = requests.post(f'{api_base}/v3/stories/search', headers=headers, json=json_data)

    print(f"🔗 code {response.status_code} on retrieving {target_state_name} stories")
    
    return response

"""
get a story's details and relationships
# 'name'
# 'description'
# 'epic_id'
# 'files'
# 'story_links # relates_to

"""


def retrieve_specific_story(story_public_id):

    story_call = requests.get(f'{api_base}/v3/stories/{story_public_id}', headers=headers)

    print(f"🔗 code {story_call.status_code} on retrieving story details for {story_public_id}")

    story = story_call.json()

    return story

"""
get an epic's details - epics often carry the PRD content a thin story description lacks

context-rich fields on an epic:
# 'name'
# 'description'
"""

def retrieve_specific_epic(epic_public_id):

    epic_call = requests.get(f'{api_base}/v3/epics/{epic_public_id}', headers=headers)

    print(f"🔗 code {epic_call.status_code} on retrieving epic details for {epic_public_id}")

    return epic_call.json()


def drop_a_comment(story, comment_payload = "", include_mentions=True):
    
    """
    inputs: 
    story = a story json obj as returned from retrieve_specific_story
    comment_payload = the comment of recommendations and asks
    
    the comment will always be a topline thread under the story
    """
    if include_mentions:
        requester_and_owners = set([story['requested_by_id']]+story['owner_ids'])
        # the agent shouldn't @ the agent
        requester_and_owners = list(i for i in requester_and_owners if i != authenticated_user_id)

        # get their mention-names to ensure notifications are properly integrated
        member_directory = requests.get(f'{api_base}/v3/members', headers=headers).json()

        to_mention = list(m['profile']['mention_name'] for m in member_directory if m['id'] in requester_and_owners)

        # mentions need a leading space/newline to parse as a mention at all - Shortcut
        # (like most platforms) won't recognise "text.@name" as a mention, only "text. @name"
        # or "text.\n\n@name" - concatenating with no separator silently drops the notification
        mentions_text = " ".join(f"@{m}" for m in to_mention)
        text = f"{comment_payload}\n\n{mentions_text}" if mentions_text else comment_payload

        json_data = {
            'text': text,  # this fires a mention/email
        }
    else:
        json_data = {
            'text': f"{comment_payload}", # this does not fire a mention/email
        }

    comment_post = requests.post(
        f'{api_base}/v3/stories/{story["id"]}/comments',
        headers=headers,
        json=json_data,
    )

    print(f"🔗 code {comment_post.status_code} on adding comment on {story['id']}")


def approval_stamp(story, approved=True):
    
    """
    Keeps all non-agentic labels attached and only touches explicitly agentic labels
    (denoted in this system by a 🤖 emoji prefix).
    """
    
    existing_labels = story['labels']
    labels_to_keep = list(l['name'] for l in existing_labels if not l['name'].startswith('🤖'))

    if approved:
        label_name = "🤖 agent approved"
    else:
        label_name = "🤖 more info needed"
        
    label_array = [{'name': l} for l in labels_to_keep] + [{'name': label_name}]
    
    label_add = requests.put(
        f'{api_base}/v3/stories/{story["id"]}',
        headers=headers,
        json={'labels': label_array},
    )
    
    print(f"🏷️ code {label_add.status_code} adding label {label_name} to {story['id']}")



In [ ]:
"""
open_linked_doc - reads a Shortcut linked_file's actual content (Google Doc, PDF, image)
rather than trusting its filename/metadata, since those can be stale or mismatched.

requires a service account JSON key, with the target file(s) shared to its client_email
(drive.readonly is enough - this tool never writes).

pip install google-api-python-client google-auth pypdf python-docx
"""


SERVICE_ACCOUNT_FILE = 'GOOGLE_SERVICE_ACCOUNT_KEY.json'  # gitignored - provide your own


DRIVE_SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

drive_credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=DRIVE_SCOPES
)
drive_service = build('drive', 'v3', credentials=drive_credentials)

MAX_FILE_SIZE_BYTES = 10 * 1_000_000  # 10MB - generous, just guards against a stray video/huge PDF

DOCX_CONTENT_TYPE = 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'


def _get_remote_size(url):
    try:
        response = requests.head(url, headers=headers, allow_redirects=True)
        content_length = response.headers.get('Content-Length')
        return int(content_length) if content_length is not None else None
    except requests.RequestException:
        return None


def _check_size(candidate):
    """
    Checked before every download, never after - a known 'size' is used as-is, an unknown
    one (embedded links) costs one HEAD request rather than a full GET blind. Returns a skip
    note if oversized, else None.
    """
    size = candidate.get('size')
    if size is None:
        size = _get_remote_size(candidate['url'])
        candidate['size'] = size  # cache it - callers (e.g. the total-download budget) reuse this

    if size is not None and size > MAX_FILE_SIZE_BYTES:
        return f"'{candidate.get('name', candidate['url'])}' is {size / 1_000_000:.1f}MB - skipped (over the {MAX_FILE_SIZE_BYTES // 1_000_000}MB limit)"

    return None


def _extract_file_id(url):
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    if not match:
        raise ValueError(f"couldn't find a file id in {url}")
    return match.group(1)


def canonical_key(entry):
    """
    Collapses duplicate references to the same file: Drive links key on the file id (catches
    query-param variants of the same url); everything else keys on 'name' rather than 'size',
    since size isn't known yet for embedded-link candidates at dedup time and can't match
    reliably against files/linked_files entries that already have it. Falls back to the raw
    url if name is missing too.
    """
    url = entry['url']
    host = url.split('/')[2] if '://' in url else ''

    if host in ('drive.google.com', 'docs.google.com'):
        try:
            return f'drive:{_extract_file_id(url)}'
        except ValueError:
            pass

    return f'name:{entry.get("name", url)}'


def _download_media_bytes(file_id):
    request = drive_service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    return buffer.read()


def _extract_content_from_bytes(raw_bytes, content_type):
    """
    Shared text/image extraction once raw bytes + content_type are known. Returns
    (content, kind), or (None, None) if unhandled. Images pass through as base64, not
    interpreted here - that's the model's job.
    """
    if content_type == 'application/pdf':
        reader = PdfReader(io.BytesIO(raw_bytes))
        return "\n".join(page.extract_text() or "" for page in reader.pages), 'text'
    elif content_type == DOCX_CONTENT_TYPE:
        doc = Document(io.BytesIO(raw_bytes))
        return "\n".join(p.text for p in doc.paragraphs), 'text'
    elif content_type and content_type.startswith('image/'):
        return base64.b64encode(raw_bytes).decode('utf-8'), content_type
    else:
        return None, None


def open_linked_doc(linked_file):
    """
    Reads a linked_files entry or a find_embedded_links candidate (content_type may be None
    for a bare Drive url with no filename to guess from). Returns (content, kind, note):
    text/pdf -> (text, 'text', None); image -> (base64, mime type, None);
    failure -> (None, None, reason). Never raises - a missing/unsupported/oversized doc is
    a finding, not a crash.
    """

    file_id = _extract_file_id(linked_file['url'])
    content_type = linked_file.get('content_type')

    try:
        if content_type is None:
            # no filename to guess from (e.g. a bare Drive URL found in prose) - ask Drive directly
            content_type = drive_service.files().get(fileId=file_id, fields='mimeType').execute()['mimeType']

        is_text_doc = content_type == 'application/vnd.google-apps.document'

        if not is_text_doc and content_type != 'application/pdf' and content_type != DOCX_CONTENT_TYPE and not content_type.startswith('image/'):
            note = f"linked file '{linked_file['name']}' is a {content_type}, which this tool doesn't read yet - couldn't verify its contents"
            print(f"⚠️ {note}")
            return None, None, note

        size_note = _check_size(linked_file)
        if size_note:
            print(f"⚠️ {size_note}")
            return None, None, size_note

        if is_text_doc:
            content = drive_service.files().export(fileId=file_id, mimeType='text/plain').execute().decode('utf-8')
            kind = 'text'
        else:
            content, kind = _extract_content_from_bytes(_download_media_bytes(file_id), content_type)

    except HttpError as e:
        status = e.resp.status
        if status == 403:
            note = f"linked file '{linked_file['name']}' isn't shared with the audit service account - couldn't verify its contents"
        elif status == 404:
            note = f"linked file '{linked_file['name']}' wasn't found (deleted or moved?) - couldn't verify its contents"
        else:
            note = f"linked file '{linked_file['name']}' couldn't be read (HTTP {status}) - couldn't verify its contents"
        print(f"⚠️ {note}")
        return None, None, note

    unit = "chars" if kind == 'text' else "bytes (base64)"
    print(f"📄 read {len(content)} {unit} from linked file '{linked_file['name']}'")

    return content, kind, None


In [4]:
"""
find_embedded_links - pulls file/doc references out of markdown/plaintext (description or
a comment) that never made it into linked_files. Returns linked_files-shaped dicts tagged
'source': 'embedded_in_text'.
"""

MARKDOWN_LINK_PATTERN = re.compile(r'(!?)\[([^\]]*)\]\((https?://[^\s)]+)\)')
BARE_URL_PATTERN = re.compile(r'https?://[^\s)]+')


def find_embedded_links(text):

    if not text:
        return []

    candidates = []
    consumed_spans = []

    for match in MARKDOWN_LINK_PATTERN.finditer(text):
        _, label, url = match.groups()
        content_type, _ = mimetypes.guess_type(label)
        candidates.append({
            'name': label or url,
            'url': url,
            'content_type': content_type,  # None if the label has no recognisable extension
            'source': 'embedded_in_text',
        })
        consumed_spans.append(match.span())

    # strip what markdown already matched so its URL isn't also picked up as a bare URL
    remaining = text
    for start, end in sorted(consumed_spans, reverse=True):
        remaining = remaining[:start] + remaining[end:]

    for match in BARE_URL_PATTERN.finditer(remaining):
        url = match.group(0)
        candidates.append({
            'name': url,
            'url': url,
            'content_type': None,  # no filename to guess from
            'source': 'embedded_in_text',
        })

    return candidates


def open_shortcut_attachment(candidate):
    """
    Downloads from Shortcut's own attachment storage (media.app.shortcut.com), using the
    same Shortcut-Token as the rest of the notebook - it's a private endpoint, not a public
    CDN. Same (content, kind, note) contract as open_linked_doc. Size is checked before the
    GET, not after - unlike open_linked_doc there's no content_type to pre-filter on, so
    this is what actually stops a large file being pulled down blind.
    """

    url = candidate['url']
    name = candidate.get('name', url)
    content_type = candidate.get('content_type') or mimetypes.guess_type(name)[0]

    size_note = _check_size(candidate)
    if size_note:
        print(f"⚠️ {size_note}")
        return None, None, size_note

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        note = f"attachment '{name}' couldn't be downloaded (HTTP {response.status_code}) - couldn't verify its contents"
        print(f"⚠️ {note}")
        return None, None, note

    content_type = content_type or response.headers.get('Content-Type', '').split(';')[0]

    if content_type and content_type.startswith('text/'):
        content, kind = response.text, 'text'
    else:
        content, kind = _extract_content_from_bytes(response.content, content_type)

    if content is None:
        note = f"attachment '{name}' is a {content_type or 'unrecognised type'}, which this tool doesn't read yet - couldn't verify its contents"
        print(f"⚠️ {note}")
        return None, None, note

    unit = "chars" if kind == 'text' else "bytes (base64)"
    print(f"📄 read {len(content)} {unit} from attachment '{name}'")

    return content, kind, None


def read_any_reference(candidate):
    """
    Routes by host: Drive urls go through open_linked_doc, everything else through
    open_shortcut_attachment. Works on both linked_files entries and find_embedded_links
    candidates - same shape.
    """

    host = candidate['url'].split('/')[2] if '://' in candidate['url'] else ''

    if host in ('drive.google.com', 'docs.google.com'):
        return open_linked_doc(candidate)

    return open_shortcut_attachment(candidate)


In [5]:
"""
get_all_referenced_files - everything attached to a story or epic, from all three places
Shortcut hides attachments: linked_files, files (native uploads - epics have neither field,
so read via .get()), and markdown-embedded links in description/comments. Not a judgment
call, so the agent shouldn't need to check each place itself. Dedupes by canonical_key,
returns one flat list ready for read_any_reference.
"""

def get_all_referenced_files(entity):

    seen_keys = set()
    references = []

    for source in ('linked_files', 'files'):
        for entry in entity.get(source, []):
            key = canonical_key(entry)
            if key not in seen_keys:
                seen_keys.add(key)
                references.append({**entry, 'source': source})

    texts = [entity['description']] + [c['text'] for c in entity['comments'] if c['text']]

    for text in texts:
        for candidate in find_embedded_links(text):
            key = canonical_key(candidate)
            if key not in seen_keys:
                seen_keys.add(key)
                references.append(candidate)

    print(f"📎 found {len(references)} referenced file(s) on {entity['entity_type']} {entity['id']}")

    return references


In [6]:
"""
gather_story_context_from_attachments / gather_epic_context_from_attachments - one tool call
for "everything referenced, already read". Both wrap _gather_context_from_attachments -
separate names so the agent's tool list stays explicit, same reasoning as
retrieve_specific_story vs retrieve_specific_epic. Also tracks a running download total
across all candidates - MAX_FILE_SIZE_BYTES alone doesn't stop many medium-sized attachments
adding up to something huge.
"""

MAX_TOTAL_DOWNLOAD_BYTES = 25 * 1_000_000  # 25MB per story/epic - a generous multiple of the
# per-file cap, so one or two normal-sized attachments never trip it, but a long tail of
# medium files (or a slow buildup over a story's history) eventually does


def _gather_context_from_attachments(entity):

    text_sections = []
    images = []
    notes = []
    total_bytes = 0

    for candidate in get_all_referenced_files(entity):

        # resolve size up front (reusing 'size' when the candidate already carries it, one
        # HEAD request otherwise) and cache it on the candidate - _check_size inside
        # read_any_reference will then see 'size' already set and skip re-fetching it
        if candidate.get('size') is None:
            candidate['size'] = _get_remote_size(candidate['url'])
        size = candidate['size']

        if size is not None and total_bytes + size > MAX_TOTAL_DOWNLOAD_BYTES:
            note = f"'{candidate['name']}' skipped - reading it would push the total download over the {MAX_TOTAL_DOWNLOAD_BYTES // 1_000_000}MB budget"
            print(f"⚠️ {note}")
            notes.append(note)
            continue

        content, kind, note = read_any_reference(candidate)

        # count it against the budget even on failure/skip past this point - a 403 or an
        # unsupported-type rejection inside open_shortcut_attachment can still happen after
        # the bytes were already pulled down, so the network cost was real either way
        if size is not None:
            total_bytes += size

        if note:
            notes.append(note)
        elif kind == 'text':
            text_sections.append(f"--- {candidate['name']} ({candidate['source']}) ---\n{content}")
        else:
            images.append({'name': candidate['name'], 'media_type': kind, 'data': content})

    return {
        'text': "\n\n".join(text_sections),
        'images': images,
        'notes': notes,
    }


def gather_story_context_from_attachments(story):
    return _gather_context_from_attachments(story)


def gather_epic_context_from_attachments(epic):
    return _gather_context_from_attachments(epic)


In [ ]:
"""
TOOL_REGISTRY - manifest of every tool above, so the system prompt can enforce an escalation
order instead of reaching for the priciest tool by default. Two axes: `tier` (read tools -
compute/token cost and how far a call strays from the story being audited; higher = rarer,
not routine) and `blast_radius` (write tools - real-world consequence, not compute cost:
does it notify a human or change shared state). Tiers are sparse on purpose - the point is
ordering, not precise cost accounting.
"""

TOOL_REGISTRY = [

    # --- discovery: outside the per-story escalation ladder, runs before it ---
    {
        'name': 'search_stories',
        'category': 'discovery',
        'tier': None,
        'cost': 'low - one API call, returns a list of story summaries',
        'use_when': "Selecting which stories need auditing in the first place. Not called "
                    "again once a specific story is being judged.",
    },

    # --- tier 1: mandatory entry point ---
    {
        'name': 'retrieve_specific_story',
        'category': 'read',
        'tier': 1,
        'cost': 'low - one API call, a few KB of JSON',
        'use_when': "Always first. The mandatory entry point for judging any story - "
                    "everything else escalates from here.",
    },

    # --- tier 2: the default reasoning surface - not a tool call, already in hand from tier 1 ---
    {
        'name': "(reason over story['description'] + story['comments'] directly)",
        'category': 'reasoning',
        'tier': 2,
        'cost': 'free - already returned by retrieve_specific_story, no extra call',
        'use_when': "Form the completeness judgment from this alone first. Most stories "
                    "should be decidable here without escalating any further.",
    },

    # --- tier 3: moderate escalation, still scoped to the story itself ---
    {
        'name': 'gather_story_context_from_attachments',
        'category': 'read',
        'tier': 3,
        'cost': 'moderate - downloads (up to 25MB), PDF/docx parsing, and real token cost '
                'once folded into context',
        'use_when': "Use when description/comments don't resolve whether a linked/embedded "
                    "file is actually a spec. This is routine, not an escalation - just don't "
                    "call it when description+comments already answer the question.",
    },

    # --- tier 4-5: expensive escalation - leaving the story's own scope, use sparingly ---
    {
        'name': 'retrieve_specific_epic',
        'category': 'read',
        'tier': 4,
        'cost': 'low API cost on its own, but represents a real escalation in scope',
        'use_when': "Use only when the story + its attachments still aren't enough. Needing "
                    "the epic to understand one story signals an upstream PRD/scoping "
                    "failure - uncommon, not routine.",
    },
    {
        'name': 'gather_epic_context_from_attachments',
        'category': 'read',
        'tier': 5,
        'cost': 'same cost as the story version, plus genuine scope creep beyond the unit '
                'being audited',
        'use_when': "The most expensive tool in the set - reserve for genuinely confusing "
                    "stories, after retrieve_specific_epic suggests the epic's description "
                    "might carry the missing spec. Needing this often is a finding about the "
                    "workspace, not routine.",
    },

    # --- write/mutating tools: scored by blast radius, not compute cost ---
    {
        'name': 'approval_stamp',
        'category': 'write',
        'blast_radius': 'low - swaps a 🤖-prefixed label only, no notification fires',
        'use_when': "Called via mark_reviewed once a verdict is reached - safe on every "
                    "audited story, the intended terminal action. mark_reviewed also posts "
                    "the reasoning as a comment - quiet if approved, notifying the requester "
                    "if not (see drop_a_comment below).",
    },
    {
        'name': 'drop_a_comment',
        'category': 'write',
        'blast_radius': "depends on include_mentions - high (notifies owners/requester) "
                         "when True, low (posted silently) when False. Defaults to True.",
        'use_when': "With mentions on: for a specific finding a human needs to see now - "
                    "never routine, never repeated. mark_reviewed already does this "
                    "automatically for a needs-more-info verdict, since that's a call to "
                    "action someone has to actually see. With mentions off: safe to use "
                    "liberally for a passive log, e.g. an approved verdict's reasoning.",
    },
]


# 🦾 the DoR agent itself

## Why state lives on the card, not in a database

**No persistent in-memory state, by design.** State belongs on-disk, not in this
process's memory - and here "on-disk" is offloaded to the Shortcut card itself (its label
and comment thread), rather than local cache.

**Benefits**

- **Auditable without extra tooling** - verdicts and reasoning live where the team
  already looks, no separate log system.
- **Crash-safe and idempotent** - a killed or duplicated run just re-derives "already
  handled" from the card. No checkpoint, no lock, no recovery logic.
- **Parallelizable** - no shared session state between stories.
- **Model/vendor portable** - swap the model or the whole implementation; state doesn't
  move, since it was never in the process to begin with.
- **No infra** - Shortcut *is* the state layer. Nothing else to run or maintain.

**Trade-offs**

- **No cross-story synthesis** - each story is judged in isolation. Spotting a pattern
  across stories (e.g. one epic with five stories missing the same thing) needs a
  separate batch-level pass, which doesn't exist yet.
- **Full re-read every run** - no cached summary, so cost scales with a story's history
  length, not just what changed.
- **Coarse re-review trigger** - any `updated_at` change re-triggers a review, even ones
  that don't matter (estimate, iteration move).
- **Race condition at scale** - two concurrent `run_audit()` runs can both miss the
  other's in-flight comment and double-post on the same story. Harmless today at one
  scheduled run; the moment this moves to multiple triggers (webhook + cron, several
  workers) it needs real locking. That's a known re-architecture item, not something the
  current design solves.
- **One source of truth, one point of failure** - the audit trail is only as durable and
  tamper-proof as Shortcut's comment stream itself.

**Design decisions**

- **Model-portable to keep costs controllable** - `MODEL_CAPABILITIES` isolates per-model quirks
  (thinking, sampling, effort), so swapping `MODEL_ID` across Claude generations doesn't
  affect the agent logic. An unrecognised model fails safe rather than 400ing.
- **Tiered tool escalation > unweighted choice** - `TOOL_REGISTRY` orders tools by cost
  (`read` tools) and blast radius (`write` tools); the system prompt is generated straight
  from it, so the doctrine can't drift from hand-written copy. The model is instructed
  to stop escalating the moment a lower tier already answers the question.
- **A hard ceiling on tool calls** - `MAX_TOOL_TURNS` limits a single confusing story so it can't stall an entire batch run.
- **Explainable verdicts** - `mark_reviewed` won't label a story without one,
  forcing the judgment to be externalized and auditable, not just a silent label flip.
- **Notification for consequence and CTAs** - an approved verdict logs quietly; a
  needs-more-info verdict notifies the requester directly, since that's the one outcome
  that's actually a call to action.


In [ ]:
"""
The agentic loop, built on the Anthropic Tool Runner (client.beta.messages.tool_runner).
pip install anthropic. Set ANTHROPIC_API_KEY before starting the kernel (not hardcoded here).

build_tools_for_story() closures bind each tool to one story, so the model only ever supplies
what it's actually deciding (comment text, verdict) - never the story dict itself. That's
also what resolves the earlier id-vs-dict inconsistency without changing the underlying
functions.
"""


MODEL_ID = "claude-sonnet-5"  # 2.5x cheaper than Opus 5 on input+output, same adaptive
                               # thinking + effort control retained - see MODEL_CAPABILITIES
MAX_TOOL_TURNS = 8  # hard ceiling on tool calls for a single story - a runaway loop on one
                     # confusing story shouldn't be able to stall the whole audit run

"""
Per-model quirks around thinking/sampling/effort - none consistent across the Claude family,
and getting one wrong 400s the request (e.g. `temperature` on a model that dropped sampling),
not a quiet no-op. Derived from MODEL_ID rather than hardcoded, since cost is real business
logic here (thinking tokens bill as output tokens; effort is the actual spend dial).

- thinking: 'default_on' (Opus 5/Sonnet 5/Fable 5/5.1 - adaptive even if omitted) |
  'explicit' (Opus 4.6/4.7/4.8, Sonnet 4.6 - must set {"type": "adaptive"} or it runs with
  none) | 'budget_tokens' (Haiku 4.5/older - no "adaptive" concept, old-style config)
- sampling: whether `temperature` is accepted at all - False wherever thinking is the
  primary mechanism (Fable 5/5.1, Opus 5, Opus 4.7/4.8, Sonnet 5), True on
  Opus 4.6/Sonnet 4.6/Haiku 4.5
- effort: whether output_config.effort is accepted - errors on Sonnet 4.5/Haiku 4.5

An unrecognised model id fails safe: no optional params sent, plus a warning - so an
unknown model still runs rather than 400ing mid-batch.
"""

MODEL_CAPABILITIES = {
    "claude-fable-5-1":  {"thinking": "default_on",    "sampling": False, "effort": True},
    "claude-mythos-5-1": {"thinking": "default_on",    "sampling": False, "effort": True},
    "claude-fable-5":    {"thinking": "default_on",    "sampling": False, "effort": True},
    "claude-opus-5":     {"thinking": "default_on",    "sampling": False, "effort": True},
    "claude-opus-4-8":   {"thinking": "explicit",      "sampling": False, "effort": True},
    "claude-opus-4-7":   {"thinking": "explicit",      "sampling": False, "effort": True},
    "claude-opus-4-6":   {"thinking": "explicit",      "sampling": True,  "effort": True},
    "claude-sonnet-5":   {"thinking": "default_on",    "sampling": False, "effort": True},
    "claude-sonnet-4-6": {"thinking": "explicit",      "sampling": True,  "effort": True},
    "claude-haiku-4-5":  {"thinking": "budget_tokens", "sampling": True,  "effort": False},
}

DEFAULT_EFFORT = "medium"   # cheaper than the API default of "high" - reasonable for an
                             # agentic loop that may run over a whole backlog, not just one story
DEFAULT_TEMPERATURE = 0     # for determinism - only sent where MODEL_CAPABILITIES allows it.
                             # even then: narrows variance, doesn't guarantee identical runs -
                             # adaptive thinking and the tool-call/result sequence both add
                             # their own variation in an agentic loop.


def build_model_kwargs(model_id):
    """
    Extra kwargs for tool_runner (thinking/temperature/output_config), included only where
    MODEL_CAPABILITIES allows. Unrecognised model -> none of these, plus a warning.
    """
    caps = MODEL_CAPABILITIES.get(model_id)

    if caps is None:
        print(f"⚠️ {model_id} isn't in MODEL_CAPABILITIES - running with no thinking/sampling/"
              f"effort overrides. Add it to the table to get deterministic, cost-tuned runs.")
        return {}

    kwargs = {}

    if caps["thinking"] in ("default_on", "explicit"):
        kwargs["thinking"] = {"type": "adaptive", "display": "summarized"}
    elif caps["thinking"] == "budget_tokens":
        kwargs["thinking"] = {"type": "enabled", "budget_tokens": 2048}  # must be < max_tokens

    if caps["sampling"]:
        kwargs["temperature"] = DEFAULT_TEMPERATURE

    if caps["effort"]:
        kwargs["output_config"] = {"effort": DEFAULT_EFFORT}

    return kwargs


# a literal template earns its keep here - "be concise" is an abstract preference a
# smaller/cheaper model can drift from turn to turn, but showing the exact shape wanted
# is something a model reliably pattern-matches against, on any model size
REASON_FORMAT_EXAMPLE = (
    "⚠️ needs more information\n\n"
    "- Missing: problem statement, acceptance criteria\n"
    "- Description is just onboarding boilerplate + a screenshot, no spec\n"
    "- Existing comment thread already shows the team is confused about scope"
)


def build_system_prompt():
    """
    Renders TOOL_REGISTRY straight into the prompt - one source of truth for the escalation
    doctrine, instead of a hand-written copy that drifts from the registry over time.
    """
    lines = [
        "You are auditing a Shortcut story for PRD completeness - is there enough here for "
        "an engineer or data scientist to build from, or does it need more information before work starts?",
        "",
        "You've already been given the story's name, description, and comments below - that "
        "alone is enough to decide most stories. Escalate through the tools below only as far "
        "as you actually need to, in tier order. Calling a higher-tier tool when a lower one "
        "already answered the question is a mistake, not thoroughness.",
        "",
        "Format any text you write for a human (`reason` or a comment) as 2-4 short markdown "
        "bullet points, never a paragraph. This is read in a story thread, not a report. "
        "Example of the shape wanted:",
        REASON_FORMAT_EXAMPLE,
        "",
    ]
    for entry in TOOL_REGISTRY:
        if entry['category'] == 'discovery':
            continue  # search_stories - not part of this per-story agent's toolset
        tier = f"tier {entry['tier']}" if entry.get('tier') is not None else entry['category']
        cost = entry.get('cost', entry.get('blast_radius'))
        lines.append(f"[{tier}] {entry['name']} - {cost}")
        lines.append(f"  use when: {entry['use_when']}")
    lines.append("")
    lines.append(
        "End every audit by calling mark_reviewed with your verdict and a short reason - "
        "the reason is required. An approved verdict logs quietly (no notification), since "
        "nothing further is needed from anyone. A needs-more-info verdict is a call to "
        "action, so it notifies the story's requester directly - that's the whole point of "
        "flagging it. Call add_comment separately only if there's a further, specific gap "
        "beyond the verdict itself that a human needs to see right now - never as a routine "
        "check-in, and never if a similar comment is already in the thread below."
    )
    return "\n".join(lines)


def build_tools_for_story(story):
    """closures bound to one story - see the cell docstring for why zero/near-zero args."""

    epic_cache = {}

    def _get_epic():
        if 'epic' not in epic_cache:
            epic_cache['epic'] = retrieve_specific_epic(story['epic_id'])
        return epic_cache['epic']

    @beta_tool
    def read_story_attachments() -> dict:
        """Read every file/doc referenced by this story (linked files, native uploads,
        embedded links) and return their extracted content. Tier 3 - only if
        description/comments alone don't establish whether a referenced file is actually
        a spec."""
        return gather_story_context_from_attachments(story)

    @beta_tool
    def read_epic() -> dict:
        """Fetch this story's parent epic (name + description). Tier 4 - only if the story
        itself still isn't enough to judge completeness. Needing this signals something
        upstream already failed."""
        return _get_epic()

    @beta_tool
    def read_epic_attachments() -> dict:
        """Read every file/doc referenced by the parent epic. Tier 5, the most expensive
        tool available - reserve for genuinely confusing stories only, after read_epic
        suggests the epic's own description might carry the missing spec."""
        return gather_epic_context_from_attachments(_get_epic())

    @beta_tool
    def add_comment(comment_text: str) -> str:
        """Post a comment on this story, notifying its owners and requester by email/mention.
        Only call this for a specific, actionable finding a human needs to see - never as a
        routine "reviewed by agent" marker, and never if a similar comment already exists.
        Format comment_text as 2-4 short markdown bullet points, never a paragraph."""
        drop_a_comment(story, comment_text)
        return "comment posted"

    @beta_tool
    def mark_reviewed(approved: bool, reason: str) -> str:
        """Record the verdict as a label, and log why as a comment. An approved verdict logs
        quietly (no notification) - nothing further is needed from anyone. A needs-more-info
        verdict notifies the story's requester instead, since it's a call to action that has
        to actually reach someone, not sit silently in the thread. Call once, as the terminal
        action. `reason` is required - never label without stating why. Format it as 2-4
        short markdown bullet points, never a paragraph - e.g. "- Missing acceptance
        criteria\\n- No problem statement, just onboarding boilerplate". This is read in a
        story thread, not a report."""
        approval_stamp(story, approved)
        verdict_line = "✅ approved - enough here to build from" if approved else "⚠️ needs more information"
        drop_a_comment(story, f"🤖 {verdict_line}\n\n{reason}", include_mentions=not approved)
        notice = "reasoning logged quietly" if approved else "requester notified"
        return f"labelled {'approved' if approved else 'needs more info'}, {notice}"

    return [read_story_attachments, read_epic, read_epic_attachments, add_comment, mark_reviewed]


def _print_reasoning(story_id, message, stopped_naturally):
    """
    Surfaces what the model actually thought and said, not just the label it left behind -
    without this, the only visible output of an audit is a silent state change in Shortcut.
    """
    status = "concluded on its own" if stopped_naturally else f"⚠️ hit the {MAX_TOOL_TURNS}-turn cap"
    print(f"--- story {story_id} {status} ---")

    if message is None:
        print("(no response)")
        return

    for block in message.content:
        if block.type == "thinking" and block.thinking:
            print(f"🧠 {block.thinking}")
        elif block.type == "text" and block.text:
            print(f"💬 {block.text}")


def run_story_audit(story):
    """
    Runs the per-story agent to completion (or MAX_TOOL_TURNS, whichever comes first).
    story['description']/['comments'] go straight into the first user message - tier 2 in
    the registry, free, no tool call needed to see them.
    """
    client = anthropic.Anthropic()

    comments_text = "\n".join(f"- {c['text']}" for c in story['comments'] if c['text']) or "(no comments)"

    user_message = (
        f"Story #{story['id']}: {story['name']}\n\n"
        f"Description:\n{story['description']}\n\n"
        f"Comments:\n{comments_text}"
    )

    runner = client.beta.messages.tool_runner(
        model=MODEL_ID,
        max_tokens=4096,
        system=build_system_prompt(),
        tools=build_tools_for_story(story),
        messages=[{"role": "user", "content": user_message}],
        **build_model_kwargs(MODEL_ID),
    )

    final_message = None
    stopped_naturally = True
    for turn, message in enumerate(runner):
        final_message = message
        if turn + 1 >= MAX_TOOL_TURNS:
            stopped_naturally = False
            break

    _print_reasoning(story['id'], final_message, stopped_naturally)

    return final_message


def needs_review(story):
    """
    Skip re-auditing if nothing's changed since the agent's own last review. Shortcut has
    no history/activity endpoint (checked against the docs), so this compares the story's
    `updated_at` (bumped by any change, silent on what) against the timestamp of this
    agent's most recent comment - which mark_reviewed now always leaves, giving every past
    review a precise marker. No prior comment -> needs review. updated_at later than that
    comment -> something changed since -> needs review. Otherwise, the last review is still
    current.
    """
    agent_comments = [
        c for c in story['comments']
        if c['text'] and c['author_id'] == authenticated_user_id
    ]

    if not agent_comments:
        return True  # never reviewed by this agent - definitely needs a look

    last_agent_review_at = max(arrow.get(c['updated_at']) for c in agent_comments)

    return arrow.get(story['updated_at']) > last_agent_review_at


def run_audit(target_state_name='To Do', days_old=10):
    """
    Outer loop: search_stories -> iterate -> run_story_audit. The only place search_stories
    is called - discovery once, judgment N times. needs_review() filters first, as a cheap
    local check before any LLM call - skipped stories cost nothing, rather than costing a
    call that the model then no-ops on.
    """
    story_summaries = search_stories(target_state_name, days_old).json()

    for summary in story_summaries:
        story = retrieve_specific_story(summary['id'])  # tier 1 - always the entry point

        if not needs_review(story):
            print(f"⏭️  story {story['id']}: nothing has changed since this agent's last review - skipping")
            continue

        print(f"\n=== auditing story {story['id']}: {story['name']} ===")
        run_story_audit(story)


# The `__main__` Loop

In [ ]:
run_audit() # run an agentic workflow